# 11. M7 and the change in MPS crime

This notebook tests whether **changes** in crime have a clearer price association than the crime level. For each sale, recent crime is the total during the 12 completed months immediately before its sale month. Earlier crime is the total during the **preceding, non-overlapping** 12 months. The change regressor is `log(1 + recent 12-month count) − log(1 + earlier 12-month count)`. A negative coefficient would associate a year-on-year rise in crime with a lower sale price.

All models use the same 317,828 sales with two complete crime years, the property/location/economic controls used in notebook 02, LSOA and year-quarter fixed effects, and LSOA-clustered standard errors. A level-only model is re-estimated on this common sample for each crime type. A supplementary two-level model estimates recent and earlier crime separately rather than imposing equal-and-opposite slopes.

The change measure may amplify reporting noise; these associations do not establish a causal effect.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.linalg import qr
from linearmodels.iv import AbsorbingLS
from IPython.display import display

ROOT=Path.cwd()
DATA=ROOT/'data'
OUT=DATA/'11_crime_mps_change_outputs'
OUT.mkdir(parents=True,exist_ok=True)
MONTHLY=DATA/'mps_monthly_panel.csv.gz'
MERGED=DATA/'crime_housing_merged.csv.gz'
BASELINE=DATA/'02_crime_mps_outputs'/'combined_primary_models.csv'
for path in [MONTHLY,MERGED,BASELINE]:
    if not path.exists(): raise FileNotFoundError(path)
MAIN=['mps_all','mps_violence','mps_property']
EXPOSURES=['annual','lagyear','prior12']
PROPERTY=['detached_dum','semi_d_dum','terrace_dum','freehold','newbuild']
LOCATION=['tube_distnear_km','bus_distnear_km','thamesriv_dist_km','dist_to_cbd','Nearest_Park_dist_km']
ECONOMIC=['grossannualpay_k','hoursworked','unemployment']
CONTROLS=PROPERTY+LOCATION+ECONOMIC
CRIME_COLS=['log1p_'+crime+'_'+exposure for exposure in EXPOSURES for crime in MAIN]
SOURCE_CRIME=[crime+'_'+exposure for exposure in EXPOSURES for crime in MAIN]

monthly=pd.read_csv(MONTHLY,usecols=['lsoacode','month']+MAIN,low_memory=False)
monthly['month']=pd.PeriodIndex(monthly.month,freq='M')
monthly=monthly.sort_values(['lsoacode','month']).reset_index(drop=True)
if monthly.duplicated(['lsoacode','month']).any():raise ValueError('Duplicate LSOA-month in MPS panel')
if (monthly.groupby('lsoacode').month.nunique()!=69).any():
    raise ValueError('Incomplete MPS month grid; inspect source coverage')
for crime in MAIN:
    monthly[crime+'_prior12_check']=monthly.groupby('lsoacode')[crime].transform(
        lambda s:s.shift(1).rolling(12,min_periods=12).sum())
    monthly[crime+'_prior24']=monthly.groupby('lsoacode')[crime].transform(
        lambda s:s.shift(1).rolling(24,min_periods=24).sum())
window=monthly[['lsoacode','month']+[c+'_prior12_check' for c in MAIN]+[c+'_prior24' for c in MAIN]]
window=window.rename(columns={'month':'sale_month'})
# The 24-month rolling count is used to form the preceding 12 months.

raw_columns=['price','age','grossannualpay','Nearest_Park_dist','tube_distnear',
             'bus_distnear','thamesriv_dist','dist_to_cbd','detached_dum','semi_d_dum',
             'terrace_dum','freehold','hoursworked','unemployment','lsoacode',
             'msoa11','oslaua','year','yearquarter','sale_month','analysis_geography_ok',
             'mps_exposures_available']+SOURCE_CRIME
data=pd.read_csv(MERGED,usecols=raw_columns,low_memory=False)
data['sale_month']=pd.PeriodIndex(data.sale_month,freq='M')
data=data.merge(window,on=['lsoacode','sale_month'],how='left',validate='many_to_one')
for crime in MAIN:
    supplied=data[crime+'_prior12']
    checked=data[crime+'_prior12_check']
    if not np.array_equal(supplied.loc[supplied.notna()].to_numpy(),
                          checked.loc[supplied.notna()].to_numpy()):
        raise AssertionError(f'24-month construction does not reproduce prior12: {crime}')
    data['log1p_'+crime+'_prior24']=np.log1p(data[crime+'_prior24'])
data['lnprice']=np.log(data.price.where(data.price.gt(0)))
data['newbuild']=data.age.eq('Y').astype(float)
data['grossannualpay_k']=data.grossannualpay/1000
data['Nearest_Park_dist_km']=data.Nearest_Park_dist/1000
for source in ['tube_distnear','bus_distnear','thamesriv_dist']:
    data[source+'_km']=data[source]/1000
for source,target in zip(SOURCE_CRIME,CRIME_COLS):
    if data[source].dropna().lt(0).any():raise ValueError('Negative crime count: '+source)
    data[target]=np.log1p(data[source])
required=['lnprice','lsoacode','msoa11','oslaua','year','yearquarter']+CONTROLS+CRIME_COLS
valid=data[required].replace('',np.nan).notna().all(axis=1)
valid &= np.isfinite(data[required].select_dtypes(include='number')).all(axis=1)
valid &= data.analysis_geography_ok & data.mps_exposures_available
baseline=pd.read_csv(BASELINE).query("model == 'M7' and exposure_label == 'prior12'")
assert len(baseline)==3 and valid.sum()==int(baseline.N.iloc[0])
matched=valid & data[[crime+'_prior24' for crime in MAIN]].notna().all(axis=1)
sample=data.loc[matched,['lnprice','lsoacode','year','yearquarter']+CONTROLS+
                ['log1p_'+crime+'_prior12' for crime in MAIN]+
                ['log1p_'+crime+'_prior24' for crime in MAIN]].copy()
coverage=pd.DataFrame({'year':sorted(data.loc[valid,'year'].unique())})
coverage['baseline_prior12_sales']=coverage.year.map(data.loc[valid].groupby('year').size()).fillna(0).astype(int)
coverage['matched_prior12_prior24_sales']=coverage.year.map(sample.groupby('year').size()).fillna(0).astype(int)
coverage['retained_share']=coverage.matched_prior12_prior24_sales/coverage.baseline_prior12_sales
coverage.to_csv(OUT/'change_sample_coverage.csv',index=False)
display(coverage)
print(f'Original prior12 sample: {valid.sum():,} sales; matched 12/24 sample: {len(sample):,} sales.')
print('Matched sale months:',data.loc[matched,'sale_month'].min(),data.loc[matched,'sale_month'].max())

# Prior 24 = recent 12 + the preceding, non-overlapping 12 months.
for crime in MAIN:
    previous=data.loc[matched,crime+'_prior24']-data.loc[matched,crime+'_prior12']
    if previous.lt(0).any() or previous.isna().any():
        raise AssertionError(f'Invalid preceding-year count for {crime}')
    sample['log1p_'+crime+'_previous12']=np.log1p(previous.to_numpy())
    sample['change_log1p_'+crime]=(
        sample['log1p_'+crime+'_prior12']-sample['log1p_'+crime+'_previous12'])
print('Change sample:',len(sample),'sales and',sample.lsoacode.nunique(),'LSOAs')


,year,baseline_prior12_sales,matched_prior12_prior24_sales,retained_share
0,2012,85375,64856,0.75966
1,2013,99429,99429,1.00000
2,2014,108207,108207,1.00000
3,2015,45336,45336,1.00000


Original prior12 sample: 338,347 sales; matched 12/24 sample: 317,828 sales.
Matched sale months: 2012-04 2015-07
Change sample: 317828 sales and 4647 LSOAs


## Match the level, change, and two-period models

The level and change models each contain one crime regressor. The two-period model includes both log crime counts, allowing the data to distinguish an association with recent crime from one with preceding-year crime. The change-only model imposes equal and opposite coefficients on the two periods; a Wald-style test of their sum checks that restriction in the unrestricted model.

In [2]:
def absorb(matrix,groups,tol=1e-10,max_iter=2000):
    z=np.asarray(matrix,dtype=float).copy()
    codes=[pd.factorize(g,sort=False)[0] for g in groups]
    if any((c<0).any() for c in codes):raise ValueError('Missing FE identifier')
    for iteration in range(max_iter):
        before=z.copy()
        for g in codes:
            counts=np.bincount(g)
            sums=np.column_stack([np.bincount(g,weights=z[:,j],minlength=len(counts))
                                  for j in range(z.shape[1])])
            z-=(sums/counts[:,None])[g]
        if np.max(np.abs(z-before))<tol:return z,iteration+1
    raise RuntimeError('Fixed-effect absorption did not converge')

exposure_cols=(['log1p_'+crime+'_prior12' for crime in MAIN]
               +['log1p_'+crime+'_previous12' for crime in MAIN]
               +['change_log1p_'+crime for crime in MAIN])
columns=['lnprice']+exposure_cols+CONTROLS
matrix=sample[columns].to_numpy(float,copy=True)
scale=matrix.std(axis=0);scale[scale<1e-14]=1
projected,iterations=absorb(matrix/scale,[sample.lsoacode,sample.yearquarter])
projected[:,projected.std(axis=0)<1e-7]=0
prepared=pd.DataFrame(projected*scale,columns=columns,index=sample.index)
G=sample.lsoacode.nunique()
critical=stats.t.ppf(.975,G-1)

def fit_spec(crime,spec,variables):
    names=variables+CONTROLS
    x=prepared[names].astype(float).copy()
    y=prepared.lnprice.astype(float).copy()
    scales=x.std(axis=0).to_numpy().copy();scales[scales<1e-14]=1
    x_scaled=x/scales
    x_scaled-=x_scaled.mean(axis=0)
    y-=y.mean()
    fit=AbsorbingLS(y,x_scaled,absorb=None,drop_absorbed=True).fit(
        cov_type='clustered',clusters=sample.lsoacode,debiased=True)
    if any(variable not in fit.params.index for variable in variables):
        raise ValueError('A crime regressor was absorbed')
    rows=[]
    for j,variable in enumerate(variables):
        beta=float(fit.params[variable])/scales[j]
        se=float(fit.std_errors[variable])/scales[j]
        rows.append(dict(model='M7',crime=crime,specification=spec,variable=variable,
                         N=len(sample),clusters=G,beta=beta,se=se,
                         p=float(2*stats.t.sf(abs(beta/se),G-1)),
                         ci_low=beta-critical*se,ci_high=beta+critical*se,
                         r2=fit.rsquared))
    restriction=None
    if len(variables)==2:
        slopes=np.array([float(fit.params[v])/scales[j] for j,v in enumerate(variables)])
        covariance=fit.cov.loc[variables,variables].to_numpy()/np.outer(scales[:2],scales[:2])
        sum_beta=float(slopes.sum())
        sum_se=float(np.sqrt(covariance.sum()))
        restriction=dict(crime=crime,N=len(sample),clusters=G,
                         recent_beta=slopes[0],preceding_beta=slopes[1],
                         beta_sum=sum_beta,beta_sum_se=sum_se,
                         equal_opposite_p=float(2*stats.t.sf(abs(sum_beta/sum_se),G-1)))
    return rows,restriction

rows=[];restriction_rows=[]
for crime in MAIN:
    recent='log1p_'+crime+'_prior12'
    earlier='log1p_'+crime+'_previous12'
    change='change_log1p_'+crime
    for spec,variables in [('recent_level',[recent]),('year_on_year_change',[change]),
                           ('recent_and_preceding',[recent,earlier])]:
        result,restriction=fit_spec(crime,spec,variables)
        rows.extend(result)
        if restriction is not None:restriction_rows.append(restriction)
results=pd.DataFrame(rows)
restriction_tests=pd.DataFrame(restriction_rows)
results.to_csv(OUT/'m7_crime_change_regressions.csv',index=False)
restriction_tests.to_csv(OUT/'m7_change_restriction_tests.csv',index=False)
display(results[['crime','specification','variable','N','beta','se','p','ci_low','ci_high']])
display(restriction_tests)


,crime,specification,variable,N,beta,se,p,ci_low,ci_high
0,mps_all,recent_level,log1p_mps_all_prior12,317828,-0.000031,0.006424,0.996131,-0.012625,0.012563
1,mps_all,year_on_year_change,change_log1p_mps_all,317828,-0.003069,0.003966,0.439141,-0.010844,0.004707
2,mps_all,recent_and_preceding,log1p_mps_all_prior12,317828,0.001156,0.006537,0.859665,-0.011660,0.013971
3,mps_all,recent_and_preceding,log1p_mps_all_previous12,317828,0.007425,0.005900,0.208272,-0.004142,0.018992
4,mps_violence,recent_level,log1p_mps_violence_prior12,317828,0.002382,0.003085,0.440115,-0.003666,0.008429
5,mps_violence,year_on_year_change,change_log1p_mps_violence,317828,0.003899,0.001790,0.029392,0.000391,0.007407
6,mps_violence,recent_and_preceding,log1p_mps_violence_prior12,317828,0.000593,0.003299,0.857332,-0.005875,0.007061
7,mps_violence,recent_and_preceding,log1p_mps_violence_previous12,317828,-0.007064,0.003144,0.024695,-0.013227,-0.000900
8,mps_property,recent_level,log1p_mps_property_prior12,317828,-0.001895,0.004907,0.699405,-0.011515,0.007725
9,mps_property,year_on_year_change,change_log1p_mps_property,317828,-0.004079,0.003172,0.198532,-0.010297,0.002139


,crime,N,clusters,recent_beta,preceding_beta,beta_sum,beta_sum_se,equal_opposite_p
0,mps_all,317828,4647,0.001156,0.007425,0.008581,0.009611,0.372005
1,mps_violence,317828,4647,0.000593,-0.007064,-0.006470,0.005356,0.227077
2,mps_property,317828,4647,-0.000699,0.007538,0.006839,0.007640,0.370719


## Plot the matched level and change estimates

Both variables are measured in log-crime units, but the change is a difference between two log counts and has a different distribution. The figure shows coefficient estimates and 95% LSOA-clustered intervals; it is not a direct test of equality between the level and change estimates.

In [3]:
labels={'mps_all':'All crime','mps_violence':'Violent crime','mps_property':'Property crime'}
fig,axes=plt.subplots(1,3,figsize=(14,4.5))
for ax,crime in zip(axes,MAIN):
    pair=results.loc[results.crime.eq(crime)&results.specification.isin(
        ['recent_level','year_on_year_change'])].set_index('specification')
    for j,(spec,colour) in enumerate([('recent_level','#2468a0'),
                                      ('year_on_year_change','#a4623c')]):
        row=pair.loc[spec]
        ax.errorbar(j,row.beta,yerr=[[row.beta-row.ci_low],[row.ci_high-row.beta]],
                    fmt='o',color=colour,capsize=4,markersize=7)
        ax.annotate(f'{row.beta:+.4f}',(j,row.beta),xytext=(0,10),
                    textcoords='offset points',ha='center',fontsize=9)
    ax.axhline(0,color='#555',linestyle='--',linewidth=.8)
    ax.set_xticks([0,1],['Recent 12-month level','Year-on-year change'])
    ax.set_xlim(-.5,1.5)
    ax.set_title(labels[crime]);ax.set_ylabel('Crime coefficient (95% clustered CI)')
    ax.grid(axis='y',alpha=.2)
fig.suptitle('M7: recent crime level versus year-on-year change, same sales',fontsize=15)
fig.tight_layout()
fig.savefig(OUT/'m7_crime_change_vs_level.png',dpi=170,bbox_inches='tight')
plt.close(fig)


## Findings and interpretation

All estimates use the same **317,828 sales** and **4,647 LSOA clusters**. The crime change is `log(1 + recent 12-month crime) − log(1 + preceding 12-month crime)`; the sale month is excluded.

| Crime type | Recent 12-month level β (p) | Year-on-year change β (p) |
|---|---:|---:|
| All crime | −0.00003 (0.996) | −0.00307 (0.439) |
| Violent crime | +0.00238 (0.440) | **+0.00390 (0.029)** |
| Property crime | −0.00189 (0.699) | −0.00408 (0.199) |

- Changing to a year-on-year crime measure does **not** reveal the expected negative association. All-crime and property-crime change coefficients are negative but imprecise. The violent-crime change coefficient is positive and its 95% confidence interval is [+0.00039, +0.00741].
- In the less restrictive two-period violent-crime model, recent crime is near zero (β = +0.00059, p = 0.857) while the preceding-year count is negative (β = −0.00706, p = 0.025). Thus the positive change coefficient must **not** be described as evidence that an increase in violent crime raises housing prices; it is a constrained combination of two period-specific associations. The equal-and-opposite restriction is not rejected (p = 0.227), but failure to reject does not validate it.
- This was one of several exploratory crime measures and exposure windows. The violent change p-value of 0.029 does not survive a simple 5% Bonferroni correction over the three crime-type change tests (threshold 0.0167). It should be reported cautiously, if at all.
- Differencing can increase measurement noise, and LSOA fixed effects already use within-area changes over time. The change transformation answers a different question about *year-on-year movement*; it is not automatically a better version of M7.

These estimates are observational and do not establish a causal price response to crime changes.
